# Proyecto Integrador — IA inmobiliaria con Random Forest, Ollama y Telegram

**Parte 1 de 3: entrenamiento del modelo de Machine Learning en Google Colab.**

Este notebook entrena un modelo `RandomForestRegressor` que estima el precio de una
vivienda de Ames (Iowa) a partir de solo tres datos:

| Dato que pide el bot | Variable original del dataset | Variable que usa el modelo |
|---|---|---|
| Sector | `Neighborhood` | `Neighborhood` |
| Metros cuadrados | `GrLivArea` (en pies cuadrados) | `Area_m2` |
| Años de antigüedad | `YrSold` - `YearBuilt` | `Antiguedad` |
| **Precio (lo que se predice)** | `SalePrice` | `SalePrice` |

**Al terminar, este notebook genera los archivos que debes descargar:**
1. `modelo_casas.pkl` — el modelo entrenado.
2. `sectores_ames.json` — el catálogo de sectores válidos.
3. `requirements_local.txt` — las versiones exactas de librerías para tu PC.

> **Advertencia académica:** el resultado es una estimación educativa basada en datos
> históricos de Ames, Iowa. No es una tasación profesional ni refleja precios de Ecuador.

---
### Cómo ejecutar este notebook
Ejecuta las celdas **en orden, de arriba hacia abajo**, con `Shift + Enter`.
No saltes celdas: cada una depende de la anterior.

## Paso 1 — Instalar e importar las librerías

**Qué hace cada librería (para la defensa oral):**
- `pandas`: trabaja con tablas de datos (filas y columnas), como un Excel programable.
- `scikit-learn`: aporta el algoritmo Random Forest y las herramientas para entrenar y evaluar.
- `joblib`: guarda el modelo ya entrenado en un archivo `.pkl` para reutilizarlo después.
- `json`: guarda el catálogo de sectores en un formato que Python y el bot pueden leer.

> **Nota sobre la instalación.** La guía sugiere ejecutar
> `!pip install -U scikit-learn pandas joblib`. **No lo hacemos**, porque la opción
> `-U` actualiza pandas a la versión 3 y el propio entorno de Colab exige la 2.2.3.
> Al romperse esa dependencia deja de funcionar `google.colab.files.download`,
> que es justo lo que necesitamos en el último paso para descargar el modelo.
>
> Colab ya trae las tres librerías preinstaladas, así que solo comprobamos qué
> versiones hay. Más adelante, `requirements_local.txt` guardará exactamente estas
> mismas versiones para que el `.pkl` funcione igual en tu PC con Windows.

In [ ]:
# Solo consultamos las versiones instaladas. No instalamos ni actualizamos nada.
!pip list | grep -E "^(scikit-learn|pandas|joblib)\s"

joblib                                1.5.3
pandas                                2.2.3
scikit-learn                          1.6.1


In [ ]:
import json

import joblib
import pandas as pd
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print("Librerias importadas correctamente.")
print("Version de scikit-learn:", sklearn.__version__)
print("Version de pandas:     ", pd.__version__)

Librerias importadas correctamente.
Version de scikit-learn: 1.6.1
Version de pandas:      2.2.3


## Paso 2 — Descargar el dataset desde OpenML

`fetch_openml` descarga automáticamente un dataset público de internet.
Usamos `data_id=42165`, que corresponde al dataset **house_prices (Ames Housing)**.

`as_frame=True` hace que los datos lleguen como una tabla de pandas en lugar de una
matriz de números sueltos.

> **Pregunta probable:** *¿Por qué no descargaste un CSV a mano?*
> Porque `fetch_openml` garantiza que siempre uso la misma versión oficial del dataset
> y el notebook es reproducible por cualquier persona sin archivos adjuntos.

In [ ]:
print("Descargando dataset desde OpenML (puede tardar unos segundos)...")

datos_openml = fetch_openml(data_id=42165, as_frame=True)
df = datos_openml.frame.copy()

print("Descarga completada.")
print("Filas:   ", df.shape[0])
print("Columnas:", df.shape[1])

df.head()

Descargando dataset desde OpenML (puede tardar unos segundos)...
Descarga completada.
Filas:    1460
Columnas: 81


,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Paso 3 — Seleccionar únicamente las variables del proyecto

El dataset original tiene **81 columnas**, pero nuestro bot solo puede preguntar 3 cosas
por Telegram. Por eso nos quedamos únicamente con las columnas que necesitamos.

> **Pregunta probable:** *¿Por qué usaste solo 3 variables si el dataset tiene 81?*
> Porque el modelo tiene que recibir exactamente los mismos datos que un usuario real
> puede escribir en un mensaje. Si entrenara con 81 variables, el bot tendría que
> preguntar 81 cosas. Sacrifico precisión a cambio de que el sistema sea usable,
> y mido cuánta precisión sacrifico con el MAE.

In [ ]:
columnas = ["Neighborhood", "GrLivArea", "YearBuilt", "YrSold", "SalePrice"]
df_modelo = df[columnas].copy()

print("Columnas seleccionadas:", columnas)
df_modelo.head()

Columnas seleccionadas: ['Neighborhood', 'GrLivArea', 'YearBuilt', 'YrSold', 'SalePrice']


,Neighborhood,GrLivArea,YearBuilt,YrSold,SalePrice
0,CollgCr,1710,2003,2008,208500
1,Veenker,1262,1976,2007,181500
2,CollgCr,1786,2001,2008,223500
3,Crawfor,1717,1915,2006,140000
4,NoRidge,2198,2000,2008,250000


## Paso 4 — Limpieza y preparación de los datos

Aquí ocurren cuatro cosas. Debes poder explicar cada una:

1. **`pd.to_numeric(..., errors="coerce")`** → convierte el texto a números.
   Si un valor no se puede convertir, lo marca como `NaN` (dato vacío) en vez de romper el programa.
2. **`dropna`** → elimina las filas que quedaron con algún dato vacío.
   No podemos entrenar con casas a las que les falta el precio o el área.
3. **`* 0.092903`** → convierte pies cuadrados a metros cuadrados
   (1 pie² = 0.092903 m²). El dataset es de EE.UU., pero el usuario piensa en metros.
4. **`YrSold - YearBuilt`** → calcula la antigüedad.
   Ejemplo: casa construida en 1990 y vendida en 2010 → tenía 20 años al venderse.

Al final descartamos las antigüedades negativas, porque una casa no puede venderse
antes de construirse: son errores del registro.

In [ ]:
# 1. Convertir a numeros las columnas que deben ser numericas
for columna in ["GrLivArea", "YearBuilt", "YrSold", "SalePrice"]:
    df_modelo[columna] = pd.to_numeric(df_modelo[columna], errors="coerce")

filas_antes = len(df_modelo)

# 2. Eliminar filas incompletas
df_modelo = df_modelo.dropna(subset=columnas).copy()

# 3. Convertir pies cuadrados a metros cuadrados
df_modelo["Area_m2"] = df_modelo["GrLivArea"] * 0.092903

# 4. Calcular la antiguedad de la vivienda al momento de la venta
df_modelo["Antiguedad"] = df_modelo["YrSold"] - df_modelo["YearBuilt"]

# 5. Descartar datos ilogicos
df_modelo = df_modelo[df_modelo["Antiguedad"] >= 0].copy()

print("Filas antes de la limpieza:  ", filas_antes)
print("Filas despues de la limpieza:", len(df_modelo))
print("Filas eliminadas:            ", filas_antes - len(df_modelo))

df_modelo[["Neighborhood", "Area_m2", "Antiguedad", "SalePrice"]].head()

Filas antes de la limpieza:   1460
Filas despues de la limpieza: 1460
Filas eliminadas:             0


,Neighborhood,Area_m2,Antiguedad,SalePrice
0,CollgCr,158.864130,5,208500
1,Veenker,117.243586,31,181500
2,CollgCr,165.924758,7,223500
3,Crawfor,159.514451,91,140000
4,NoRidge,204.200794,8,250000


In [ ]:
# Un vistazo rapido a los datos ya limpios (util para la diapositiva 3)
df_modelo[["Area_m2", "Antiguedad", "SalePrice"]].describe().round(2)

## Paso 5 — Definir `X` (las entradas) e `y` (la respuesta)

Esta es **una de las preguntas seguras del profesor**. La respuesta corta:

- **`X` son las preguntas.** Lo que el modelo conoce: sector, metros y antigüedad.
- **`y` es la respuesta.** Lo que el modelo debe aprender a estimar: el precio.

Entrenar el modelo es mostrarle miles de pares (X, y) hasta que aprenda la relación
entre ambos. Después le damos una X nueva y él nos devuelve su y estimada.

> **¿Qué significa que sea un problema de regresión?**
> Que la respuesta `y` es un **número continuo** (un precio: 184.350, 207.900...).
> Si en cambio tuviéramos que responder "cara" o "barata", sería un problema de clasificación.

In [ ]:
X = df_modelo[["Neighborhood", "Area_m2", "Antiguedad"]].copy()
y = df_modelo["SalePrice"].copy()

print("X - lo que el modelo recibe:")
print(X.head())
print()
print("y - lo que el modelo debe predecir:")
print(y.head())

X - lo que el modelo recibe:
  Neighborhood     Area_m2  Antiguedad
0      CollgCr  158.864130           5
1      Veenker  117.243586          31
2      CollgCr  165.924758           7
3      Crawfor  159.514451          91
4      NoRidge  204.200794           8

y - lo que el modelo debe predecir:
0    208500
1    181500
2    223500
3    140000
4    250000
Name: SalePrice, dtype: int64


## Paso 6 — Separar datos de entrenamiento y de prueba

Dividimos en **80 % para entrenar** y **20 % para probar**.

> **Pregunta probable:** *¿Por qué dividimos los datos?*
> Porque si evalúo el modelo con las mismas casas que usó para aprender, estaría
> midiendo su memoria, no su capacidad de estimar. El 20 % de prueba son casas que el
> modelo **nunca vio**: es el examen honesto.

`random_state=42` fija el azar de la división, para que cada vez que ejecute el
notebook obtenga exactamente los mismos resultados. Es lo que hace el experimento
reproducible.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Casas para ENTRENAR:", X_train.shape[0])
print("Casas para PROBAR:  ", X_test.shape[0])

Casas para ENTRENAR: 1168
Casas para PROBAR:   292


## Paso 7 — Convertir el sector (texto) a números con OneHotEncoder

**Pregunta casi garantizada: ¿por qué el sector necesita One Hot Encoding?**

Porque Random Forest solo trabaja con números, y `Neighborhood` es texto (`"OldTown"`,
`"NAmes"`, `"Gilbert"`...).

La tentación sería numerarlos: OldTown = 1, NAmes = 2, Gilbert = 3. **Eso está mal**,
porque el modelo entendería que Gilbert es "más" que OldTown, o que NAmes está justo en
medio de los dos. Y eso no significa nada: son barrios, no una escala.

OneHotEncoder crea **una columna por cada sector**, con 1 si la casa está ahí y 0 si no:

| | es_OldTown | es_NAmes | es_Gilbert |
|---|---|---|---|
| Casa en OldTown | **1** | 0 | 0 |
| Casa en NAmes | 0 | **1** | 0 |

Así ningún barrio vale más que otro: simplemente se pertenece o no se pertenece.

`handle_unknown="ignore"` evita que el programa se caiga si algún día llega un sector
que no estaba en los datos de entrenamiento.

`passthrough` significa "estas dos columnas ya son números, déjalas pasar tal cual".

In [ ]:
preprocesador = ColumnTransformer(
    transformers=[
        ("sector", OneHotEncoder(handle_unknown="ignore"), ["Neighborhood"]),
        ("numericas", "passthrough", ["Area_m2", "Antiguedad"])
    ]
)

print("Preprocesador configurado.")
print("- 'Neighborhood' se convertira en columnas de 0 y 1 (OneHotEncoder).")
print("- 'Area_m2' y 'Antiguedad' pasan tal cual porque ya son numeros.")

Preprocesador configurado.
- 'Neighborhood' se convertira en columnas de 0 y 1 (OneHotEncoder).
- 'Area_m2' y 'Antiguedad' pasan tal cual porque ya son numeros.


## Paso 8 — Crear y entrenar el Random Forest

**¿Qué hace un Random Forest en términos sencillos?**

Imagina que le preguntas el precio de una casa a 80 tasadores distintos. Cada uno
revisó una parte diferente del mercado y se fijó en aspectos distintos. Random Forest
construye esos 80 "tasadores" (**árboles de decisión**), cada uno da su precio, y el
resultado final es el **promedio de los 80**.

¿Por qué es mejor que un solo árbol? Porque un tasador solo puede equivocarse mucho por
un sesgo suyo. Al promediar 80 opiniones independientes, los errores individuales se
cancelan entre sí.

**Los parámetros, en cristiano:**
- `n_estimators=80` → cuántos árboles. Más árboles = mejor, pero más lento.
- `max_depth=10` → cuántas preguntas seguidas puede hacer cada árbol. Si es muy alto,
  el árbol se memoriza los datos (*sobreajuste*) en vez de aprender la regla general.
- `min_samples_leaf=2` → cada conclusión debe apoyarse en al menos 2 casas reales.
  Evita que el modelo saque reglas a partir de un caso aislado.
- `n_jobs=-1` → usa todos los núcleos del procesador para ir más rápido.

**¿Por qué un Pipeline?** Porque une el preprocesamiento y el modelo en un solo objeto.
Así, cuando guarde el `.pkl`, el archivo llevará dentro **el encoder y el bosque juntos**.
El bot podrá pasarle el sector como texto (`"OldTown"`) sin tener que recrear el
OneHotEncoder a mano. Es lo que hace que la conexión con Telegram sea limpia.

In [ ]:
random_forest = RandomForestRegressor(
    n_estimators=80,
    max_depth=10,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

modelo = Pipeline(steps=[
    ("preprocesamiento", preprocesador),
    ("random_forest", random_forest)
])

print("Entrenando el modelo... (esto es el 'aprendizaje')")
modelo.fit(X_train, y_train)
print("Modelo entrenado correctamente.")

Entrenando el modelo... (esto es el 'aprendizaje')
Modelo entrenado correctamente.


## Paso 9 — Evaluar el modelo: MAE, RMSE y R²

Estas tres métricas van **obligatoriamente en tu presentación**. Así se explican:

- **MAE (Error Absoluto Medio)** → *"en promedio, mi modelo se equivoca en X dólares"*.
  Es la más fácil de explicar y la más honesta ante un cliente.

- **RMSE (Raíz del Error Cuadrático Medio)** → parecido al MAE, pero **castiga más los
  errores grandes**, porque los eleva al cuadrado antes de promediar.
  Si el RMSE es mucho mayor que el MAE, significa que el modelo acierta casi siempre
  pero de vez en cuando se equivoca muchísimo (normalmente en las casas de lujo).

- **R² (Coeficiente de determinación)** → va de 0 a 1 y responde:
  *"¿qué porcentaje de la variación de los precios logra explicar mi modelo?"*
  Un R² de 0.75 significa que las 3 variables explican el 75 % de por qué una casa
  cuesta más que otra. El resto depende de cosas que no le di al modelo: número de
  baños, garaje, calidad de los acabados, estado de la cocina...

> **Si el profesor dice que el error es alto:** tiene razón, y es esperable, porque el
> modelo solo conoce 3 de las 81 variables disponibles. Es una decisión de diseño
> consciente: el sistema tiene que funcionar con lo que un usuario escribe en un
> mensaje de Telegram. La mejora obvia sería pedir más datos en la conversación.

In [ ]:
predicciones = modelo.predict(X_test)

mae = mean_absolute_error(y_test, predicciones)
rmse = mean_squared_error(y_test, predicciones) ** 0.5
r2 = r2_score(y_test, predicciones)

print("METRICAS DEL MODELO (sobre las casas que nunca vio)")
print("---------------------------------------------------")
print(f"MAE:  ${mae:,.2f}")
print(f"RMSE: ${rmse:,.2f}")
print(f"R2:   {r2:.3f}")
print()
print("INTERPRETACION PARA LA PRESENTACION:")
print(f"- En promedio el modelo se equivoca en ${mae:,.0f} por casa.")
print(f"- El modelo explica el {r2*100:.1f}% de la variacion de los precios")
print("  usando solo sector, metros cuadrados y antiguedad.")

METRICAS DEL MODELO (sobre las casas que nunca vio)
---------------------------------------------------
MAE:  $24,043.85
RMSE: $36,414.53
R2:   0.827

INTERPRETACION PARA LA PRESENTACION:
- En promedio el modelo se equivoca en $24,044 por casa.
- El modelo explica el 82.7% de la variacion de los precios
  usando solo sector, metros cuadrados y antiguedad.


### Guarda esta captura
Toma un **screenshot de la celda anterior**: esas tres cifras son la evidencia
de la diapositiva 5 de tu PowerPoint.

## Paso 10 — Probar una predicción manual

Esto simula exactamente lo que hará el bot: recibir sector, metros y antigüedad,
y devolver un precio.

Fíjate en que le pasamos `"OldTown"` **como texto**. El Pipeline se encarga solo de
convertirlo con el OneHotEncoder. Esa es la ventaja de haber usado Pipeline.

In [ ]:
ejemplo = pd.DataFrame([{
    "Neighborhood": "OldTown",
    "Area_m2": 120,
    "Antiguedad": 25
}])

precio_estimado = modelo.predict(ejemplo)[0]

print("Datos de entrada:")
print(ejemplo)
print()
print(f"Precio estimado: ${precio_estimado:,.0f}")

Datos de entrada:
  Neighborhood  Area_m2  Antiguedad
0      OldTown      120          25

Precio estimado: $171,393


In [ ]:
# Comparacion rapida de varios sectores con la misma casa.
# Sirve para demostrar que el sector SI influye en el precio.
casas_de_prueba = pd.DataFrame([
    {"Neighborhood": "OldTown", "Area_m2": 120, "Antiguedad": 25},
    {"Neighborhood": "NAmes",   "Area_m2": 120, "Antiguedad": 25},
    {"Neighborhood": "Gilbert", "Area_m2": 120, "Antiguedad": 25},
    {"Neighborhood": "NridgHt", "Area_m2": 120, "Antiguedad": 25},
])

casas_de_prueba["precio_estimado"] = modelo.predict(casas_de_prueba).round(0)
casas_de_prueba

,Neighborhood,Area_m2,Antiguedad,precio_estimado
0,OldTown,120,25,171393.0
1,NAmes,120,25,171009.0
2,Gilbert,120,25,169177.0
3,NridgHt,120,25,178207.0


## Paso 11 — Guardar el modelo y los archivos para el bot

Aquí generamos los archivos que necesita tu PC con Windows.

**Importante — el catálogo de sectores.** No guardamos una simple lista de códigos,
sino un diccionario `{"OldTown": "Old Town", "NAmes": "North Ames", ...}`.

¿Por qué? Porque el usuario escribirá *"Old Town"* o *"North Ames"* en lenguaje natural,
pero el modelo fue entrenado con los códigos `OldTown` y `NAmes`. Ese diccionario es el
**catálogo de traducción que le pasamos a Mistral** dentro del prompt, para que devuelva
siempre el código correcto. Sin él, Mistral inventaría variantes y la validación fallaría.

**`requirements_local.txt`** guarda la versión exacta de scikit-learn de Colab. Es
crítico: si en tu PC instalas otra versión, al cargar el `.pkl` puede fallar o dar
avisos. Instalando desde este archivo, ambas versiones coinciden.

In [ ]:
# Nombres legibles de los sectores de Ames, para que Mistral pueda traducir
# lo que escribe el usuario ("Old Town") al codigo del dataset ("OldTown").
NOMBRES_SECTORES = {
    "Blmngtn": "Bloomington Heights",
    "Blueste": "Bluestem",
    "BrDale":  "Briardale",
    "BrkSide": "Brookside",
    "ClearCr": "Clear Creek",
    "CollgCr": "College Creek",
    "Crawfor": "Crawford",
    "Edwards": "Edwards",
    "Gilbert": "Gilbert",
    "IDOTRR":  "Iowa DOT and Rail Road",
    "MeadowV": "Meadow Village",
    "Mitchel": "Mitchell",
    "NAmes":   "North Ames",
    "NoRidge": "Northridge",
    "NPkVill": "Northpark Villa",
    "NridgHt": "Northridge Heights",
    "NWAmes":  "Northwest Ames",
    "OldTown": "Old Town",
    "SWISU":   "South and West of Iowa State University",
    "Sawyer":  "Sawyer",
    "SawyerW": "Sawyer West",
    "Somerst": "Somerset",
    "StoneBr": "Stone Brook",
    "Timber":  "Timberland",
    "Veenker": "Veenker",
}

# Solo incluimos los sectores que realmente quedaron tras la limpieza.
# Si apareciera un codigo sin nombre conocido, usamos el propio codigo.
codigos_presentes = sorted(df_modelo["Neighborhood"].unique().tolist())
sectores_ames = {
    codigo: NOMBRES_SECTORES.get(codigo, codigo)
    for codigo in codigos_presentes
}

print("Sectores incluidos:", len(sectores_ames))
for codigo, nombre in sectores_ames.items():
    print(f"  {codigo:10s} -> {nombre}")

Sectores incluidos: 25
  Blmngtn    -> Bloomington Heights
  Blueste    -> Bluestem
  BrDale     -> Briardale
  BrkSide    -> Brookside
  ClearCr    -> Clear Creek
  CollgCr    -> College Creek
  Crawfor    -> Crawford
  Edwards    -> Edwards
  Gilbert    -> Gilbert
  IDOTRR     -> Iowa DOT and Rail Road
  MeadowV    -> Meadow Village
  Mitchel    -> Mitchell
  NAmes      -> North Ames
  NPkVill    -> Northpark Villa
  NWAmes     -> Northwest Ames
  NoRidge    -> Northridge
  NridgHt    -> Northridge Heights
  OldTown    -> Old Town
  SWISU      -> South and West of Iowa State University
  Sawyer     -> Sawyer
  SawyerW    -> Sawyer West
  Somerst    -> Somerset
  StoneBr    -> Stone Brook
  Timber     -> Timberland
  Veenker    -> Veenker


In [ ]:
# 1. El modelo entrenado completo (preprocesamiento + Random Forest)
joblib.dump(modelo, "modelo_casas.pkl")

# 2. El catalogo de sectores para el prompt de Mistral
with open("sectores_ames.json", "w", encoding="utf-8") as archivo:
    json.dump(sectores_ames, archivo, ensure_ascii=False, indent=2)

# 3. Las versiones exactas de librerias, para que el .pkl funcione en Windows
with open("requirements_local.txt", "w", encoding="utf-8") as archivo:
    archivo.write(f"scikit-learn=={sklearn.__version__}\n")
    archivo.write(f"pandas=={pd.__version__}\n")
    archivo.write("joblib\n")
    archivo.write("requests\n")

# 4. Las metricas, para no tener que volver a entrenar si las necesitas
with open("metricas_modelo.json", "w", encoding="utf-8") as archivo:
    json.dump({
        "MAE": round(float(mae), 2),
        "RMSE": round(float(rmse), 2),
        "R2": round(float(r2), 4),
        "filas_entrenamiento": int(X_train.shape[0]),
        "filas_prueba": int(X_test.shape[0]),
        "variables": ["Neighborhood", "Area_m2", "Antiguedad"],
        "sklearn_version": sklearn.__version__,
    }, archivo, ensure_ascii=False, indent=2)

print("Archivos generados correctamente:")
print("  - modelo_casas.pkl")
print("  - sectores_ames.json")
print("  - requirements_local.txt")
print("  - metricas_modelo.json")

Archivos generados correctamente:
  - modelo_casas.pkl
  - sectores_ames.json
  - requirements_local.txt
  - metricas_modelo.json


## Paso 12 — Verificar que el `.pkl` funciona antes de descargarlo

Volvemos a cargar el archivo desde cero y hacemos una predicción con él.
Si esto funciona, el bot también funcionará.

**Este paso te ahorra el peor escenario:** descubrir el día de la demostración que el
archivo estaba corrupto o incompleto.

In [ ]:
modelo_recargado = joblib.load("modelo_casas.pkl")

with open("sectores_ames.json", "r", encoding="utf-8") as archivo:
    sectores_recargados = json.load(archivo)

prueba = pd.DataFrame([{
    "Neighborhood": "OldTown",
    "Area_m2": 120.0,
    "Antiguedad": 25
}])

precio = modelo_recargado.predict(prueba)[0]

print("VERIFICACION FINAL")
print("------------------")
print("Sectores cargados:", len(sectores_recargados))
print("Tipo de dato de sectores:", type(sectores_recargados).__name__, "(debe decir 'dict')")
print(f"Prediccion de prueba: ${precio:,.0f}")
print()
print("Si ves un precio arriba, el modelo esta listo para el bot.")

VERIFICACION FINAL
------------------
Sectores cargados: 25
Tipo de dato de sectores: dict (debe decir 'dict')
Prediccion de prueba: $171,393

Si ves un precio arriba, el modelo esta listo para el bot.


## Paso 13 — Descargar los archivos a tu computadora

Ejecuta la celda siguiente. Colab te va a descargar los 4 archivos uno por uno
(el navegador puede pedirte permiso para descargas múltiples: acepta).

**Después colócalos en la carpeta:**
`C:\Users\hidal\OneDrive\Desktop\bot_ia_inmobiliaria`

Si la descarga automática falla, usa el panel de archivos de la izquierda
(el ícono de la carpeta), clic derecho sobre cada archivo → **Descargar**.

In [ ]:
from google.colab import files

for nombre_archivo in [
    "modelo_casas.pkl",
    "sectores_ames.json",
    "requirements_local.txt",
    "metricas_modelo.json",
]:
    files.download(nombre_archivo)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---
## Resumen de lo que hiciste en este notebook

1. Descargaste casas reales de Ames, Iowa, desde OpenML.
2. Te quedaste con 3 variables de entrada y 1 variable objetivo.
3. Limpiaste los datos, convertiste pies² a m² y calculaste la antigüedad.
4. Separaste 80 % para entrenar y 20 % para evaluar.
5. Convertiste el sector de texto a números con OneHotEncoder.
6. Entrenaste un bosque de 80 árboles de decisión.
7. Mediste el error real con MAE, RMSE y R².
8. Exportaste el modelo para usarlo desde el bot de Telegram.

**Siguiente parte:** `bot_telegram_ollama.py`, donde Mistral leerá el mensaje del
usuario, extraerá estos 3 datos y se los entregará a este modelo.

> Recuerda: **Mistral nunca calcula el precio.** Solo traduce lenguaje natural a datos.
> El único que produce un número es este Random Forest.